<a href="https://colab.research.google.com/github/Amisha-55/Analysing-Survival-on-the-Titanic/blob/main/V4_Hybrid%20Recommender/V4_Hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import kagglehub
import os

path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")

print("Dataset path:", path)
print("\nFiles:")
print(os.listdir(path))

Using Colab cache for faster access to the 'tmdb-movie-metadata' dataset.
Dataset path: /kaggle/input/tmdb-movie-metadata

Files:
['tmdb_5000_movies.csv', 'tmdb_5000_credits.csv']


In [12]:
import pandas as pd
import ast

movies_raw = pd.read_csv(
    os.path.join(path, "tmdb_5000_movies.csv")
)

credits = pd.read_csv(
    os.path.join(path, "tmdb_5000_credits.csv")
)

print("Movies:", movies_raw.shape)
print("Credits:", credits.shape)

Movies: (4803, 20)
Credits: (4803, 4)


In [13]:
import zipfile
import urllib.request
import os
import ssl

unverified_context = ssl._create_unverified_context()

url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
zip_path = "/content/ml-1m.zip"

with urllib.request.urlopen(url, context=unverified_context) as response, \
     open(zip_path, 'wb') as out_file:
    out_file.write(response.read())

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content/")

print("Dataset downloaded and extracted!")

Dataset downloaded and extracted!


In [14]:
movies_ml = pd.read_csv(
    "/content/ml-1m/movies.dat",
    sep="::",
    engine="python",
    names=["movie_id", "title", "genres"],
    encoding="latin-1"
)

print("MovieLens movies:", movies_ml.shape)
display(movies_ml.head())

MovieLens movies: (3883, 3)


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [15]:
# Clean titles for matching

movies_ml["match_title"] = (
    movies_ml["title"]
    .str.replace(r"\s*\(\d{4}\)$", "", regex=True)
    .str.lower()
    .str.strip()
)

movies_raw["match_title"] = (
    movies_raw["title"]
    .str.lower()
    .str.strip()
)

# Match MovieLens movies with TMDB movies
movie_mapping = movies_ml.merge(
    movies_raw[["id", "title", "match_title"]],
    on="match_title",
    how="inner",
    suffixes=("_ml", "_tmdb")
)

print("MovieLens movies:", len(movies_ml))
print("Matched movies:", len(movie_mapping))

print(
    "MovieLens coverage:",
    f"{len(movie_mapping) / len(movies_ml) * 100:.2f}%"
)

display(
    movie_mapping[
        ["movie_id", "title_ml", "id", "title_tmdb"]
    ].head(15)
)

MovieLens movies: 3883
Matched movies: 892
MovieLens coverage: 22.97%


,movie_id,title_ml,id,title_tmdb
0,1,Toy Story (1995),862,Toy Story
1,10,GoldenEye (1995),710,GoldenEye
2,14,Nixon (1995),10858,Nixon
3,15,Cutthroat Island (1995),1408,Cutthroat Island
4,16,Casino (1995),524,Casino
5,17,Sense and Sensibility (1995),4584,Sense and Sensibility
6,18,Four Rooms (1995),5,Four Rooms
7,19,Ace Ventura: When Nature Calls (1995),9273,Ace Ventura: When Nature Calls
8,20,Money Train (1995),11517,Money Train
9,21,Get Shorty (1995),8012,Get Shorty


In [16]:
print(movies_raw.shape)
print(movies_raw.columns.tolist())

display(
    movies_raw[["id", "title", "release_date"]].head()
)

(4803, 21)
['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'match_title']


,id,title,release_date
0,19995,Avatar,2009-12-10
1,285,Pirates of the Caribbean: At World's End,2007-05-19
2,206647,Spectre,2015-10-26
3,49026,The Dark Knight Rises,2012-07-16
4,49529,John Carter,2012-03-07


In [17]:
print(movies_ml[["movie_id", "title"]].head())

   movie_id                               title
0         1                    Toy Story (1995)
1         2                      Jumanji (1995)
2         3             Grumpier Old Men (1995)
3         4            Waiting to Exhale (1995)
4         5  Father of the Bride Part II (1995)


In [18]:
# Extract release year from MovieLens titles
movies_ml["year"] = movies_ml["title"].str.extract(
    r"\((\d{4})\)"
).astype(float)

# Extract release year from TMDB
movies_raw["tmdb_year"] = pd.to_datetime(
    movies_raw["release_date"],
    errors="coerce"
).dt.year

# Normalize titles
movies_ml["clean_title"] = (
    movies_ml["title"]
    .str.replace(r"\s*\(\d{4}\)$", "", regex=True)
    .str.lower()
    .str.replace(r"[^a-z0-9 ]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

movies_raw["clean_title"] = (
    movies_raw["title"]
    .str.lower()
    .str.replace(r"[^a-z0-9 ]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Match using BOTH title and year
movie_mapping = movies_ml.merge(
    movies_raw[
        ["id", "title", "clean_title", "tmdb_year"]
    ],
    left_on=["clean_title", "year"],
    right_on=["clean_title", "tmdb_year"],
    how="inner"
)

print("MovieLens movies:", len(movies_ml))
print("Matched movies:", len(movie_mapping))
print(
    "Coverage:",
    f"{len(movie_mapping) / len(movies_ml) * 100:.2f}%"
)

display(
    movie_mapping[
        ["movie_id", "title_x", "year", "id", "title_y"]
    ].head(20)
)

MovieLens movies: 3883
Matched movies: 793
Coverage: 20.42%


,movie_id,title_x,year,id,title_y
0,1,Toy Story (1995),1995.0,862,Toy Story
1,10,GoldenEye (1995),1995.0,710,GoldenEye
2,14,Nixon (1995),1995.0,10858,Nixon
3,15,Cutthroat Island (1995),1995.0,1408,Cutthroat Island
4,16,Casino (1995),1995.0,524,Casino
5,17,Sense and Sensibility (1995),1995.0,4584,Sense and Sensibility
6,18,Four Rooms (1995),1995.0,5,Four Rooms
7,19,Ace Ventura: When Nature Calls (1995),1995.0,9273,Ace Ventura: When Nature Calls
8,20,Money Train (1995),1995.0,11517,Money Train
9,21,Get Shorty (1995),1995.0,8012,Get Shorty


In [19]:
!pip install rapidfuzz -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.5 MB/s eta 0:00:00


In [20]:
from rapidfuzz import process, fuzz

# Create lookup of TMDB titles
tmdb_titles = movies_raw["clean_title"].dropna().unique()

matches = []

for _, row in movies_ml.iterrows():

    title = row["clean_title"]

    if not title:
        continue

    match = process.extractOne(
        title,
        tmdb_titles,
        scorer=fuzz.token_sort_ratio
    )

    if match is None:
        continue

    matched_title, score, _ = match

    # Accept strong title matches
    if score >= 90:

        tmdb_match = movies_raw[
            movies_raw["clean_title"] == matched_title
        ].iloc[0]

        matches.append({
            "movie_id": row["movie_id"],
            "ml_title": row["title"],
            "ml_year": row["year"],
            "tmdb_id": tmdb_match["id"],
            "tmdb_title": tmdb_match["title"],
            "tmdb_year": tmdb_match["tmdb_year"],
            "match_score": score
        })

movie_mapping_fuzzy = pd.DataFrame(matches)

print("Fuzzy matched movies:", len(movie_mapping_fuzzy))
print(
    "Coverage:",
    f"{len(movie_mapping_fuzzy) / len(movies_ml) * 100:.2f}%"
)

Fuzzy matched movies: 1220
Coverage: 31.42%


In [21]:
display(
    movie_mapping_fuzzy[
        [
            "movie_id",
            "ml_title",
            "ml_year",
            "tmdb_id",
            "tmdb_title",
            "tmdb_year",
            "match_score"
        ]
    ].head(20)
)

,movie_id,ml_title,ml_year,tmdb_id,tmdb_title,tmdb_year,match_score
0,1,Toy Story (1995),1995.0,862,Toy Story,1995.0,100.0
1,10,GoldenEye (1995),1995.0,710,GoldenEye,1995.0,100.0
2,11,"American President, The (1995)",1995.0,9087,The American President,1995.0,100.0
3,14,Nixon (1995),1995.0,10858,Nixon,1995.0,100.0
4,15,Cutthroat Island (1995),1995.0,1408,Cutthroat Island,1995.0,100.0
5,16,Casino (1995),1995.0,524,Casino,1995.0,100.0
6,17,Sense and Sensibility (1995),1995.0,4584,Sense and Sensibility,1995.0,100.0
7,18,Four Rooms (1995),1995.0,5,Four Rooms,1995.0,100.0
8,19,Ace Ventura: When Nature Calls (1995),1995.0,9273,Ace Ventura: When Nature Calls,1995.0,100.0
9,20,Money Train (1995),1995.0,11517,Money Train,1995.0,100.0


In [22]:
print(
    movie_mapping_fuzzy["match_score"].describe()
)

count    1220.000000
mean       99.712560
std         1.434242
min        90.000000
25%       100.000000
50%       100.000000
75%       100.000000
max       100.000000
Name: match_score, dtype: float64


In [23]:
# Keep only high-confidence matches
movie_mapping_final = movie_mapping_fuzzy[
    movie_mapping_fuzzy["match_score"] >= 95
].copy()

# Remove duplicate MovieLens IDs
movie_mapping_final = (
    movie_mapping_final
    .drop_duplicates("movie_id")
)

print("Shared movies:", len(movie_mapping_final))

display(
    movie_mapping_final[
        ["movie_id", "ml_title", "tmdb_id", "tmdb_title", "match_score"]
    ].head(10)
)

Shared movies: 1177


,movie_id,ml_title,tmdb_id,tmdb_title,match_score
0,1,Toy Story (1995),862,Toy Story,100.0
1,10,GoldenEye (1995),710,GoldenEye,100.0
2,11,"American President, The (1995)",9087,The American President,100.0
3,14,Nixon (1995),10858,Nixon,100.0
4,15,Cutthroat Island (1995),1408,Cutthroat Island,100.0
5,16,Casino (1995),524,Casino,100.0
6,17,Sense and Sensibility (1995),4584,Sense and Sensibility,100.0
7,18,Four Rooms (1995),5,Four Rooms,100.0
8,19,Ace Ventura: When Nature Calls (1995),9273,Ace Ventura: When Nature Calls,100.0
9,20,Money Train (1995),11517,Money Train,100.0


In [24]:
import pandas as pd

# Load the ratings data
ratings = pd.read_csv(
    "/content/ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

print("MovieLens ratings:", ratings.shape)
display(ratings.head())

ratings_hybrid = ratings.merge(
    movie_mapping_final[
        ["movie_id", "tmdb_id"]
    ],
    on="movie_id",
    how="inner"
)

print("Hybrid ratings:", ratings_hybrid.shape)
print(
    "Users:",
    ratings_hybrid["user_id"].nunique()
)
print(
    "Movies:",
    ratings_hybrid["tmdb_id"].nunique()
)


MovieLens ratings: (1000209, 4)


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


Hybrid ratings: (588130, 5)
Users: 6040
Movies: 1145


In [25]:
!pip install scikit-surprise -q

from surprise import Dataset, Reader, SVD

reader = Reader(
    rating_scale=(1, 5)
)

surprise_data = Dataset.load_from_df(
    ratings_hybrid[
        ["user_id", "tmdb_id", "rating"]
    ],
    reader
)

trainset_hybrid = surprise_data.build_full_trainset()

hybrid_svd = SVD(
    n_factors=100,
    n_epochs=20,
    reg_all=0.05,
    random_state=42
)

hybrid_svd.fit(trainset_hybrid)

print("Hybrid SVD trained successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 22.8 MB/s eta 0:00:00
Hybrid SVD trained successfully.


In [26]:
import kagglehub

tmdb_path = kagglehub.dataset_download(
    "tmdb/tmdb-movie-metadata"
)

print("TMDB path:", tmdb_path)
print(os.listdir(tmdb_path))

Using Colab cache for faster access to the 'tmdb-movie-metadata' dataset.
TMDB path: /kaggle/input/tmdb-movie-metadata
['tmdb_5000_movies.csv', 'tmdb_5000_credits.csv']


In [27]:
movies_raw = pd.read_csv(
    os.path.join(tmdb_path, "tmdb_5000_movies.csv")
)

credits = pd.read_csv(
    os.path.join(tmdb_path, "tmdb_5000_credits.csv")
)

print("TMDB movies:", movies_raw.shape)

TMDB movies: (4803, 20)


In [28]:
import pandas as pd
tmdb_index = pd.Series(
    movies_raw.index,
    index=movies_raw["id"]
)

shared_tmdb_ids = movie_mapping_final["tmdb_id"].unique()

print("Shared TMDB movies:", len(shared_tmdb_ids))

Shared TMDB movies: 1151


In [29]:
shared_indices = [
    tmdb_index[movie_id]
    for movie_id in shared_tmdb_ids
    if movie_id in tmdb_index
]

print("Content vectors available:", len(shared_indices))

Content vectors available: 1151


In [30]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Map TMDB movie ID → TF-IDF row
tmdb_to_index = pd.Series(
    movies_raw.index,
    index=movies_raw["id"]
)

def hybrid_recommend(movie_title, user_id, n=10, alpha=0.5):

    # Find selected movie
    matches = movies_raw[
        movies_raw["title"].str.lower() == movie_title.lower()
    ]

    if matches.empty:
        return "Movie not found."

    movie_id = matches.iloc[0]["id"]

    if movie_id not in tmdb_to_index:
        return "Movie not available in hybrid dataset."

    movie_idx = tmdb_to_index[movie_id]

    # Candidate movies
    candidates = movie_mapping_final["tmdb_id"].unique()

    candidates = [
        x for x in candidates
        if x != movie_id
    ]

    # Content similarity
    content_scores = cosine_similarity(
        tfidf_matrix[movie_idx],
        tfidf_matrix
    ).flatten()

    results = []

    for candidate_id in candidates:

        idx = tmdb_to_index[candidate_id]

        content_score = content_scores[idx]

        # SVD predicted rating
        svd_prediction = hybrid_svd.predict(
            user_id,
            candidate_id
        ).est

        results.append({
            "tmdb_id": candidate_id,
            "content_score": content_score,
            "svd_score": svd_prediction
        })

    results = pd.DataFrame(results)

    # Normalize scores
    results["content_norm"] = (
        (results["content_score"] - results["content_score"].min())
        /
        (results["content_score"].max() -
         results["content_score"].min() + 1e-8)
    )

    results["svd_norm"] = (
        (results["svd_score"] - results["svd_score"].min())
        /
        (results["svd_score"].max() -
         results["svd_score"].min() + 1e-8)
    )

    # Hybrid score
    results["hybrid_score"] = (
        alpha * results["content_norm"]
        +
        (1 - alpha) * results["svd_norm"]
    )

    # Get titles
    results = results.merge(
        movies_raw[["id", "title"]],
        left_on="tmdb_id",
        right_on="id",
        how="left"
    )

    return (
        results
        .sort_values("hybrid_score", ascending=False)
        [["title", "content_norm", "svd_norm", "hybrid_score"]]
        .head(n)
        .reset_index(drop=True)
    )

In [32]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Make sure tags exist
print("Tags available:", "tags" in movies_raw.columns)

Tags available: False


In [33]:
import ast

def convert_to_text(value):
    try:
        items = ast.literal_eval(value)
        return " ".join(
            item["name"] for item in items
        )
    except:
        return ""

movies_raw["genres_text"] = movies_raw["genres"].apply(convert_to_text)
movies_raw["keywords_text"] = movies_raw["keywords"].apply(convert_to_text)

movies_raw["tags"] = (
    movies_raw["genres_text"] + " " +
    movies_raw["keywords_text"] + " " +
    movies_raw["overview"].fillna("")
)

In [34]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(
    movies_raw["tags"]
)

print("TF-IDF matrix:", tfidf_matrix.shape)

TF-IDF matrix: (4803, 5000)


In [35]:
hybrid_recommend(
    "Inception",
    user_id=4169,
    n=10
)
hybrid_recommend(
    "The Dark Knight Rises",
    user_id=4169,
    n=10
)
hybrid_recommend(
    "The Dark Knight Rises",
    user_id=4169,
    n=10
)

,title,content_norm,svd_norm,hybrid_score
0,Batman,0.990088,0.590767,0.790428
1,Batman Returns,1.000000,0.439433,0.719716
2,Batman Forever,0.974281,0.323122,0.648702
3,L.A. Confidential,0.193257,0.889029,0.541143
4,Metropolis,0.179275,0.861098,0.520186
5,Schindler's List,0.032102,1.000000,0.516051
6,The Godfather,0.088958,0.926342,0.507650
7,On the Waterfront,0.091674,0.917638,0.504656
8,Boys Don't Cry,0.108338,0.900600,0.504469
9,The Usual Suspects,0.121338,0.886493,0.503915


In [36]:
weights = [0.2, 0.4, 0.5, 0.6, 0.8]

for alpha in weights:

    result = hybrid_recommend(
        "Inception",
        user_id=4169,
        n=5,
        alpha=alpha
    )

    print(f"\nα = {alpha}")
    display(result[["title", "hybrid_score"]])


α = 0.2


,title,hybrid_score
0,Much Ado About Nothing,0.913431
1,Casablanca,0.846205
2,Schindler's List,0.832616
3,The Best Years of Our Lives,0.823224
4,To Kill a Mockingbird,0.793432



α = 0.4


,title,hybrid_score
0,Much Ado About Nothing,0.935074
1,Happiness,0.777781
2,The Best Years of Our Lives,0.762399
3,Dangerous Liaisons,0.726603
4,Casablanca,0.706416



α = 0.5


,title,hybrid_score
0,Much Ado About Nothing,0.945895
1,Happiness,0.794702
2,The Best Years of Our Lives,0.731986
3,Dangerous Liaisons,0.704291
4,Flatliners,0.703574



α = 0.6


,title,hybrid_score
0,Much Ado About Nothing,0.956716
1,Happiness,0.811623
2,Flatliners,0.739522
3,The Best Years of Our Lives,0.701573
4,The Fisher King,0.696529



α = 0.8


,title,hybrid_score
0,Much Ado About Nothing,0.978358
1,Happiness,0.845465
2,Flatliners,0.811419
3,Entrapment,0.746306
4,The Replacements,0.728884


In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load ratings if needed
ratings = pd.read_csv(
    "/content/ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

# Train/Test split
train, test = train_test_split(
    ratings,
    test_size=0.20,
    random_state=42
)

# Relevant movies in test set (rating >= 4)
test_relevant = test[
    test["rating"] >= 4
]

# User → relevant movies
test_relevant_by_user = (
    test_relevant
    .groupby("user_id")["movie_id"]
    .apply(set)
    .to_dict()
)

# Select evaluation users
evaluation_users = list(
    test_relevant_by_user.keys()
)[:200]

print("Train ratings:", len(train))
print("Test ratings:", len(test))
print("Relevant users:", len(test_relevant_by_user))
print("Evaluation users:", len(evaluation_users))

Train ratings: 800167
Test ratings: 200042
Relevant users: 5994
Evaluation users: 200


In [50]:
#Step 10: Evaluate α values
#10.1 Create content similarity matrix
from sklearn.metrics.pairwise import cosine_similarity

shared_tmdb_ids = movie_mapping_final["tmdb_id"].unique()

shared_indices = [
    tmdb_to_index[mid]
    for mid in shared_tmdb_ids
]

shared_similarity = cosine_similarity(
    tfidf_matrix[shared_indices]
)

print("Similarity matrix:", shared_similarity.shape)

Similarity matrix: (1151, 1151)


In [38]:
#10.2 Create a fast ID → index lookup
shared_id_to_idx = {
    movie_id: i
    for i, movie_id in enumerate(shared_tmdb_ids)
}

In [49]:
from math import log2
alphas = [0.2, 0.4, 0.5, 0.6, 0.8]

results = []

# MovieLens ID → TMDB ID
ml_to_tmdb = dict(
    zip(
        movie_mapping_final["movie_id"],
        movie_mapping_final["tmdb_id"]
    )
)

# Shared TMDB movies
shared_movies = set(
    movie_mapping_final["tmdb_id"]
)

for alpha in alphas:

    precision_list = []
    recall_list = []
    ndcg_list = []

    for user_id in evaluation_users:

        relevant_ml = test_relevant_by_user[user_id]

        # Convert relevant MovieLens IDs → TMDB IDs
        relevant = {
            ml_to_tmdb[m]
            for m in relevant_ml
            if m in ml_to_tmdb
        }

        if not relevant:
            continue

        # Movies user rated in training
        rated_ml = set(
            train[
                train["user_id"] == user_id
            ]["movie_id"]
        )

        rated_tmdb = {
            ml_to_tmdb[m]
            for m in rated_ml
            if m in ml_to_tmdb
        }

        candidates = list(
            shared_movies - rated_tmdb
        )

        scores = []

        for movie_id in candidates:

            # SVD score
            svd_score = hybrid_svd.predict(
                user_id,
                movie_id
            ).est

            # Content preference:
            # similarity to movies the user rated highly
            user_history = list(rated_tmdb)

            if user_history:

                similarities = [
                    shared_similarity[
                        shared_id_to_idx[h],
                        shared_id_to_idx[movie_id]
                    ]
                    for h in user_history
                    if h in shared_id_to_idx
                ]

                content_score = max(similarities) if similarities else 0

            else:
                content_score = 0

            scores.append(
                (
                    movie_id,
                    content_score,
                    svd_score
                )
            )

        df = pd.DataFrame(
            scores,
            columns=[
                "movie_id",
                "content",
                "svd"
            ]
        )

        # Normalize both signals
        for col in ["content", "svd"]:

            mn = df[col].min()
            mx = df[col].max()

            df[col + "_norm"] = (
                (df[col] - mn)
                / (mx - mn + 1e-8)
            )

        # Hybrid score
        df["hybrid"] = (
            alpha * df["content_norm"]
            +
            (1 - alpha) * df["svd_norm"]
        )

        top10 = df.nlargest(
            10,
            "hybrid"
        )["movie_id"].tolist()

        hits = [
            m for m in top10
            if m in relevant
        ]

        # Precision
        precision_list.append(
            len(hits) / 10
        )

        # Recall
        recall_list.append(
            len(hits) / len(relevant)
        )

        # NDCG
        dcg = sum(
            1 / log2(i + 2)
            for i, m in enumerate(top10)
            if m in relevant
        )

        ideal = min(len(relevant), 10)

        idcg = sum(
            1 / log2(i + 2)
            for i in range(ideal)
        )

        ndcg_list.append(
            dcg / idcg if idcg else 0
        )

    results.append({
        "Alpha": alpha,
        "Precision@10": np.mean(precision_list),
        "Recall@10": np.mean(recall_list),
        "NDCG@10": np.mean(ndcg_list)
    })

hybrid_results = pd.DataFrame(results)

display(hybrid_results)

,Alpha,Precision@10,Recall@10,NDCG@10
0,0.2,0.098477,0.084396,0.131853
1,0.4,0.098985,0.089369,0.134503
2,0.5,0.087310,0.081289,0.124824
3,0.6,0.079188,0.075750,0.115809
4,0.8,0.067005,0.066007,0.092987


In [52]:
best_alpha = 0.4

hybrid_config = {
    "alpha": best_alpha,
    "content_weight": best_alpha,
    "svd_weight": 1 - best_alpha
}

print("V4 Configuration:")
print(hybrid_config)

V4 Configuration:
{'alpha': 0.4, 'content_weight': 0.4, 'svd_weight': 0.6}
